In [42]:
import pandas as pd
from pandas.api.types import CategoricalDtype

In [67]:
df = pd.read_csv("../medias/fusion_tawictr_branche_propre.csv", sep=";", low_memory=False)

In [68]:
COLUMNS_TO_KEEP = ['AWICTR_CODBCH', 'LIBELLE_BRANCHE', 'FIN_NUMTIE', 'MOIS_SAISIE',
       'AWICTR_DATSAICTR', 'AWICTR_CLEPAR', 'AWICTR_CODPRD', 'AWICTR_CODANM',
       'AWICTR_NUMASR', 'AWICTR_CODSPD', 'AWICTR_USRSASSOU', 'AWICTR_CMRCTX',
       'AWICTR_CTSANNHT']

In [69]:
df2 = df[COLUMNS_TO_KEEP]
df2.AWICTR_DATSAICTR = pd.to_datetime(df2.AWICTR_DATSAICTR, format="%d%b%Y", errors="raise")

#categories_branches = CategoricalDtype(categories=df2.AWICTR_CODBCH.unique().astype(str))
#df2.AWICTR_CODBCH=df2.AWICTR_CODBCH.astype(categories_branches)

df2 = df2.dropna(subset="AWICTR_DATSAICTR")

In [70]:
df2.dtypes

AWICTR_CODBCH                int64
LIBELLE_BRANCHE                str
FIN_NUMTIE                 float64
MOIS_SAISIE                float64
AWICTR_DATSAICTR    datetime64[us]
AWICTR_CLEPAR                int64
AWICTR_CODPRD                  str
AWICTR_CODANM              float64
AWICTR_NUMASR              float64
AWICTR_CODSPD                  str
AWICTR_USRSASSOU               str
AWICTR_CMRCTX                int64
AWICTR_CTSANNHT            float64
dtype: object

In [71]:
df2.head()

,AWICTR_CODBCH,LIBELLE_BRANCHE,FIN_NUMTIE,MOIS_SAISIE,AWICTR_DATSAICTR,AWICTR_CLEPAR,AWICTR_CODPRD,AWICTR_CODANM,AWICTR_NUMASR,AWICTR_CODSPD,AWICTR_USRSASSOU,AWICTR_CMRCTX,AWICTR_CTSANNHT
0,1,AUTOMOB,NaN,6.0,2003-06-12,1,AB,NaN,NaN,M5,NaN,1,235.57
1,1,AUTOMOB,73.0,11.0,2019-11-13,1,AA,0.0,80524188.0,1A,U241530,0,344.40
2,1,AUTOMOB,13.0,9.0,2008-09-12,1,AB,0.0,2353956.0,P5,C4824BN,0,637.98
3,1,AUTOMOB,38.0,4.0,2009-04-15,1,AC,0.0,2697397.0,NaN,C2526LX,0,20.21
4,1,AUTOMOB,37.0,4.0,2019-04-10,1,AA,0.0,4340887.0,1A,LI12405,0,0.00


In [72]:
df2.describe()

,AWICTR_CODBCH,FIN_NUMTIE,MOIS_SAISIE,AWICTR_DATSAICTR,AWICTR_CLEPAR,AWICTR_CODANM,AWICTR_NUMASR,AWICTR_CMRCTX,AWICTR_CTSANNHT
count,47403.000000,43364.000000,47403.000000,47403,47403.000000,43364.0,4.014000e+04,47403.000000,4.740300e+04
mean,3.261734,49.876095,6.223762,2012-02-15 18:53:11.038541,3.239373,0.0,3.908836e+07,0.043668,2.386874e+02
min,1.000000,0.000000,1.000000,1968-07-01 00:00:00,1.000000,0.0,0.000000e+00,0.000000,0.000000e+00
25%,2.000000,25.000000,3.000000,2007-11-30 00:00:00,2.000000,0.0,4.869336e+06,0.000000,0.000000e+00
50%,3.000000,50.000000,6.000000,2013-06-19 00:00:00,3.000000,0.0,2.117187e+07,0.000000,0.000000e+00
75%,5.000000,75.000000,9.000000,2017-04-21 00:00:00,5.000000,0.0,7.070852e+07,0.000000,1.497700e+02
max,12.000000,99.000000,12.000000,2022-01-31 00:00:00,5.000000,0.0,3.000004e+08,16.000000,3.197520e+06
std,1.421027,28.845222,3.397357,NaN,1.360745,0.0,3.988228e+07,0.251639,1.492716e+04


In [73]:
df2.AWICTR_CODBCH.value_counts()

AWICTR_CODBCH
2     15611
5     11886
4     11270
3      4541
1      3989
12      106
Name: count, dtype: int64

AWICTR_CODBCH (branche) avec les variables suivantes en sortie
AWICTR_CODBCH, LIBELLE_BRANCHE_FMT, FIRST_DATE, LAST_DATE,
DATE_INF_ECART, DATE_SUP_ECART, ECART_JOUR_MAX, NB_CTR où :

In [226]:
res = []
for name, group in df2.groupby("AWICTR_CODBCH"):
    group = group.sort_values(by="AWICTR_DATSAICTR")
    group = group.reset_index()
    group["date_decalee"] = group.AWICTR_DATSAICTR.shift(periods=1) 
    group["ecarts"] = group.AWICTR_DATSAICTR  - group["date_decalee"] 
    index_de_lecart_max = group[group.ecarts == group.ecarts.max()].index
    group[group.ecarts == group.ecarts.max()]
    dates_ecart_max = group.loc[[index_de_lecart_max.start, index_de_lecart_max.start - 1]].AWICTR_DATSAICTR
    la_date_ecart_max, la_date_ecart_min = dates_ecart_max

    res.append({"AWICTR_CODBCH": name, "LIBELLE_BRANCHE_FMT":group.LIBELLE_BRANCHE.unique(), 
          "FIRST_DATE":min(group.AWICTR_DATSAICTR), "LAST_DATE": max(group.AWICTR_DATSAICTR), 
           "DATE_INF_ECART":la_date_ecart_min, "DATE_SUP_ECART": la_date_ecart_max, "NB_CTR": group.shape[0]
          })

    """
    print(name)
    print(group.LIBELLE_BRANCHE.unique())
    print(min(group.AWICTR_DATSAICTR))
    print(max(group.AWICTR_DATSAICTR))

    print(dates_ecart_max)
    print(df.shape[0])
    """

In [229]:
res = pd.DataFrame(res)
res["ecart"] = res.DATE_SUP_ECART - res.DATE_INF_ECART
res

,AWICTR_CODBCH,LIBELLE_BRANCHE_FMT,FIRST_DATE,LAST_DATE,DATE_INF_ECART,DATE_SUP_ECART,NB_CTR,ecart
0,1,[AUTOMOB],1979-05-15,2022-01-31,1980-05-22,1982-10-14,3989,875 days
1,2,[IARD ET],1977-03-10,2022-01-31,1979-01-01,1981-01-29,15611,759 days
2,3,[PREV.SA],1968-07-01,2022-01-24,1968-07-01,1972-07-01,4541,1461 days
3,4,[PREV. D],1984-04-01,2022-01-27,1984-04-01,1985-01-19,11270,293 days
4,5,[CREDITS],1978-01-01,2022-01-26,1978-01-01,1986-04-29,11886,3040 days
5,12,[CAUTION],2006-07-27,2019-10-09,2013-07-12,2014-06-16,106,339 days


In [224]:
res = []
for branche, groupe in df2.groupby("AWICTR_CODBCH"):
    dates = groupe["AWICTR_DATSAICTR"].sort_values().reset_index(drop=True)
    ecarts = dates.diff()                     # date - date précédente
    ligne = {"AWICTR_CODBCH": branche,
             "LIBELLE_BRANCHE_FMT": groupe["LIBELLE_BRANCHE"].iloc[0],
             "FIRST_DATE": dates.iloc[0], "LAST_DATE": dates.iloc[-1],
             "DATE_INF_ECART": pd.NaT, "DATE_SUP_ECART": pd.NaT,
             "ECART_JOUR_MAX": 0, "NB_CTR": len(dates)}
    if len(dates) > 1:                        # un seul contrat : pas d'écart
        i = ecarts.idxmax()                   # 1re ligne du plus grand écart
        ligne.update(DATE_INF_ECART=dates[i - 1], DATE_SUP_ECART=dates[i],
                     ECART_JOUR_MAX=ecarts[i].days)
    res.append(ligne)
branches_cotisations = pd.DataFrame(res)

In [225]:
branches_cotisations

,AWICTR_CODBCH,LIBELLE_BRANCHE_FMT,FIRST_DATE,LAST_DATE,DATE_INF_ECART,DATE_SUP_ECART,ECART_JOUR_MAX,NB_CTR
0,1,AUTOMOB,1979-05-15,2022-01-31,1980-05-22,1982-10-14,875,3989
1,2,IARD ET,1977-03-10,2022-01-31,1979-01-01,1981-01-29,759,15611
2,3,PREV.SA,1968-07-01,2022-01-24,1968-07-01,1972-07-01,1461,4541
3,4,PREV. D,1984-04-01,2022-01-27,1984-04-01,1985-01-19,293,11270
4,5,CREDITS,1978-01-01,2022-01-26,1978-01-01,1986-04-29,3040,11886
5,12,CAUTION,2006-07-27,2019-10-09,2013-07-12,2014-06-16,339,106
